In [1]:
from pathlib import Path

p = Path("./data").glob("*.*")
files = [x.name for x in p if x.is_file()]
files.sort()

files

['.DS_Store',
 'AlexisMoulin-SQ6388FJ-30x-WGS-Sequencing_com-05-19-26.1.fq.gz',
 'AlexisMoulin-SQ6388FJ-30x-WGS-Sequencing_com-05-19-26.2.fq.gz',
 'AlexisMoulin-SQ6388FJ-30x-WGS-Sequencing_com-05-19-26.cnv.vcf.gz',
 'AlexisMoulin-SQ6388FJ-30x-WGS-Sequencing_com-05-19-26.snp-indel.genome.vcf.gz',
 'AlexisMoulin-SQ6388FJ-30x-WGS-Sequencing_com-05-19-26.sv.vcf.gz']

In [2]:
from cyvcf2 import VCF
import pandas as pd

In [3]:
vcf_path = "./data/AlexisMoulin-SQ6388FJ-30x-WGS-Sequencing_com-05-19-26.snp-indel.genome.vcf.gz"

vcf = VCF(vcf_path)

In [4]:
print("Samples:", vcf.samples)

for v in vcf:
    print(v.CHROM, v.POS, v.REF, v.ALT, v.genotypes[:1])
    break

Samples: ['SQ6388FJ']
1 10001 T [] [[0, 0, False]]


In [5]:
rows = []

for v in VCF(fname=vcf_path):
    rows.append({
        "chrom": v.CHROM,
        "pos": v.POS,
        "ref": v.REF,
        "alt": ",".join(v.ALT),
        "qual": v.QUAL,
        "filter": v.FILTER or "PASS",
        "type": v.var_type,
        "genotype": v.genotypes[0][:2]
    })

df = pd.DataFrame(rows)

In [6]:
df.head()

,chrom,pos,ref,alt,qual,filter,type,genotype
0,1,10001,T,,100.699997,PASS,unknown,"[0, 0]"
1,1,10250,A,C,59.020000,PASS,snp,"[0, 1]"
2,1,10251,C,,147.300003,PASS,unknown,"[0, 0]"
3,1,10279,T,C,24.990000,PASS,snp,"[0, 1]"
4,1,10280,A,,182.039993,PASS,unknown,"[0, 0]"


In [7]:
df.shape

(11805658, 8)

In [8]:
from collections import Counter

counts = Counter()

for v in vcf:
    counts["total"] += 1
    counts[v.var_type] += 1

    if v.FILTER is None:
        counts["PASS"] += 1
    else:
        counts["filtered"] += 1

print(counts)

Counter({'total': 11805657, 'PASS': 11181021, 'unknown': 6697418, 'snp': 4147303, 'indel': 960936, 'filtered': 624636})
